In [ ]:
import subprocess

import numpy as np
import pandas as pd

In [ ]:
output = subprocess.run(
    f"mysql -t -u test_user -ptest123 < test_employees_sha.sql",
    cwd="./test_db",
    capture_output=True,
    text=True,
    shell=True,
).stdout.split("\n")

In [ ]:
output

In [ ]:
expected = dict()
found = dict()
match = dict()

output_clean = []
for row in output:
    if row.startswith("+"):
        continue
    mod_row = []
    for col in row.strip().split("|"):
        if col == "":
            continue
        mod_row.append(col.strip())

    if len(mod_row) < 3:
        continue

    output_clean.append(mod_row)

table_section = None
for row in output_clean:
    if (
        row[1].startswith("expected")
        and row[2].startswith("expected")
        and table_section != "expected"
    ):
        table_section = "expected"
        continue
    elif (
        row[1].startswith("found")
        and row[2].startswith("found")
        and table_section != "found"
    ):
        table_section = "found"
        continue
    elif (
        row[1].endswith("match")
        and row[2].endswith("match")
        and table_section != "match"
    ):
        table_section = "match"
        continue

    match table_section:
        case "expected":
            expected[row[0]] = [int(row[1]), row[2]]
        case "found":
            found[row[0]] = [int(row[1]), row[2]]
        case "match":
            match[row[0]] = [row[1].upper() == "OK", row[2].upper() == "OK"]

all_match = bool(np.all(list(match.values())))

record_df = pd.DataFrame(
    {
        "table_name": list(found.keys()),
        "expected_records": np.array(list(expected.values()))[:, 0].tolist(),
        "found_records": np.array(list(found.values()))[:, 0].tolist(),
        "records_match": np.array(list(match.values()))[:, 0].tolist(),
    }
)

crc_df = pd.DataFrame(
    {
        "table_name": list(found.keys()),
        "expected_crc": np.array(list(expected.values()))[:, 1].tolist(),
        "found_crc": np.array(list(found.values()))[:, 1].tolist(),
        "crc_match": np.array(list(match.values()))[:, 1].tolist(),
    }
)

if not all_match:
    raise AssertionError(
        "The Database Integrity test failed!\n\n"
        "-----------------------------------\n"
        "----------- TEST OUTPUT -----------\n"
        f"-----------------------------------\n\n{record_df}\n\n{crc_df}"
    )

print(
    "-----------------------------------\n"
    "----------- TEST OUTPUT -----------\n"
    f"-----------------------------------\n\n{record_df}\n\n{crc_df}"
)